📦 Bloque 0 — Imports estándar

In [ ]:
import pandas as pd
import numpy as np
import sys
import importlib
import json
from os import path
from pathlib import Path

🧩 Bloque 1 — Import módulo `functions` + carga config JSON

In [ ]:
# --- Resolver rutas ---
PROJECT_ROOT = Path.cwd().resolve().parents[1]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()
CONFIG_DIR = (PROJECT_ROOT / "config").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:", CONFIG_DIR, "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
module_name = "functions"

try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)

    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))

except Exception as e:
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"sys.path{sys.path[0] if sys.path else '<vacío>'}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "config_volvek_planes.json")

try:
    with open(config_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()) if isinstance(data, dict) else type(data))

except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")


⚙️ Bloque 2 — Conexión SQL Server

In [ ]:
# --- Parámetros desde config ---
server = data["server_config"]["server"]
database = data["server_config"]["database"]
schema = data["server_config"]["schema"]

driver = "ODBC Driver 17 for SQL Server"

# 1) ODBC connection string (pyodbc)
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# 2) SQLAlchemy engine
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",
)

📂 Bloque 3 — Rutas y hojas preferidas

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "CCLA" / "VOLVEK PLANES").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

EXTS = (".xlsx",)

PREFERRED_SHEETS = ["Base Planes", "planes", "Planes"]
FALLBACK_SHEET_INDEX = 0

📑 Bloque 4 — Detección automática de archivo y hoja + preview

In [ ]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen   = info["archivo_origen"]
excel_path_used  = info["excel_path_used"]
_tmp_copy_path   = info["tmp_copy_path"]
target           = info["target_sheet"]
sheet_names      = info["sheet_names"]

# (Opcional) preview sin header
df_opexcel = fun.read_excel_safe_no_header(excel_path_used, target)
fun.pretty_table(df_opexcel, n=5)

📄 Archivo más reciente: Base cesantia SURA Planes NOVIEMBRE 2025.xlsx  |  2025-12-29 16:34:55
Hojas: ['Hoja1', 'Planes']
✅ Hoja objetivo: Planes


🧾 Bloque 5 — Lectura + normalización de headers

In [ ]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = fun.read_excel_safe(io, target)

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "none": "", "NaN": "", "nan": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]
print("Encabezados normalizados:", list(df_raw.columns))
fun.pretty_table(df_raw, n=5)

Encabezados normalizados: ['noperacion', 'afinom', 'afirut', 'afirutdv', 'crecuotot', 'cresolmon', 'fecinicob', 'fectercob', 'crecuomon', 'prima_cliente', 'tasa', 'plan', 'prima_exenta', 'comision_caja_25', 'producto', 'folio_original', 'fecha_otorgamiento_folio_original']


,noperacion,afinom,afirut,afirutdv,crecuotot,cresolmon,fecinicob,fectercob,crecuomon,prima_cliente,tasa,plan,prima_exenta,comision_caja_25,producto,folio_original,fecha_otorgamiento_folio_original
0,136000203058,DAFNE AGUILERA CAMPOS,16692442,1,37,2132125,20240822,20270930,50581,2324,0.109,Plan 4 Cuotas,1952.9411764705883,488.2352941176471,REPROGRAMACION,136000189359,20180524
1,134000381606,JUAN ORTIZ QUINTANA,17621935,1,60,9580551,20250701,20300731,141283,18490,0.193,Plan 8 Cuotas,15537.81512605042,3884.453781512605,REPROGRAMACION,134000365938,20180326
2,041000403541,PATRICIO GONZALEZ CASTILLO,9349593,4,57,6773967,20210531,20260228,125063,13097,0.193,Plan 8 Cuotas,11005.882352941177,2751.470588235294,REPROGRAMACION,241000034673,20170821
3,001027701039,GERMAN NARANJO SANDOVAL,10157539,K,60,2027725,20240619,20290630,48100,3914,0.193,Plan 8 Cuotas,3289.075630252101,822.2689075630252,REPROGRAMACION,008000518808,20180419
4,085000200303,PATRIC MOLINA VIILLAGRAN,19514790,6,88,3887153,20180731,20251130,73983,9813,0.252,Plan 12 Cuotas,8246.218487394959,2061.5546218487398,REPROGRAMACION,001027663597,20170830


🗂️ Bloque 6 — Mapeo de columnas (Origen → Destino)

In [ ]:
# Origen → Destino
foliocredito        = fun.pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio")
NombreAfiliado      = fun.pick(df_raw, "afinom", "nombre_afiliado", "nombre")
rutafiliado         = fun.pick(df_raw, "afirut", "rut_afiliado", "rut")
dvafiliado          = fun.pick(df_raw, "afirutdv", "dv_afiliado", "dv")
Plazo               = fun.pick(df_raw, "crecuotot", "plazo")
MontoBruto          = fun.pick(df_raw, "cresolmon", "monto_bruto", "monto")
fechaPrimerVto      = fun.pick(df_raw, "fecinicob", "fecha_primer_vto", "fec_ini_cob")
FechaUltimoVto      = fun.pick(df_raw, "fectercob", "fecha_ultimo_vto", "fec_ter_cob")
ValorCuota          = fun.pick(df_raw, "crecuomon", "valor_cuota")
PrimaBruta_src      = fun.pick(df_raw, "prima_cliente", "prima_bruta", "prima_cliente_")
Tasa                = fun.pick(df_raw, "tasa_interes", "tasa")
Planes              = fun.pick(df_raw, "plan")
PrimaNeta_src       = fun.pick(df_raw, "prima_exenta", "prima_neta")
Com25_src           = fun.pick(df_raw, "comision_caja_25", "comision_caja_25_", "comision_25")
Com26_src           = fun.pick(df_raw, "comision_caja_variable_39", "comision_variable_39", "comision_39")
Producto            = fun.pick(df_raw, "producto")
FolioOrigen         = fun.pick(df_raw, "folio_original", "folio_origen")
FechaOrigen         = fun.pick(df_raw, "fecha_otorgamiento_folio_original", "fecha_origen")

df_can = pd.DataFrame({
    "foliocredito": foliocredito,
    "NombreAfiliado": NombreAfiliado,
    "rutafiliado": rutafiliado,
    "dvafiliado": dvafiliado,
    "Plazo": Plazo,
    "MontoBruto": MontoBruto,
    "fechaPrimerVto": fechaPrimerVto,
    "FechaUltimoVto": FechaUltimoVto,
    "ValorCuota": ValorCuota,
    "PrimaBruta": PrimaBruta_src,
    "Tasa": Tasa,
    "Planes": Planes,
    "PrimaNeta": PrimaNeta_src,
    "Comision25": Com25_src,
    "ComisionVariable": Com26_src,
    "Producto": Producto,
    "FolioOrigen": FolioOrigen,
    "FechaOrigen": FechaOrigen,
    "Nombre_de_archivo": archivo_origen.name,
})

fun.pretty_table(df_can, n=5)

,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,Tasa,Planes,PrimaNeta,Comision25,ComisionVariable,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
0,136000203058,DAFNE AGUILERA CAMPOS,16692442,1,37,2132125,20240822,20270930,50581,2324,0.109,Plan 4 Cuotas,1952.9411764705883,488.2352941176471,None,REPROGRAMACION,136000189359,20180524,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
1,134000381606,JUAN ORTIZ QUINTANA,17621935,1,60,9580551,20250701,20300731,141283,18490,0.193,Plan 8 Cuotas,15537.81512605042,3884.453781512605,None,REPROGRAMACION,134000365938,20180326,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
2,041000403541,PATRICIO GONZALEZ CASTILLO,9349593,4,57,6773967,20210531,20260228,125063,13097,0.193,Plan 8 Cuotas,11005.882352941177,2751.470588235294,None,REPROGRAMACION,241000034673,20170821,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
3,001027701039,GERMAN NARANJO SANDOVAL,10157539,K,60,2027725,20240619,20290630,48100,3914,0.193,Plan 8 Cuotas,3289.075630252101,822.2689075630252,None,REPROGRAMACION,008000518808,20180419,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
4,085000200303,PATRIC MOLINA VIILLAGRAN,19514790,6,88,3887153,20180731,20251130,73983,9813,0.252,Plan 12 Cuotas,8246.218487394959,2061.5546218487398,None,REPROGRAMACION,001027663597,20170830,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx


🏷️ Bloque 7 — Canonicalización nombre de archivo (SURA Planes YYYYMM)

In [ ]:
nombre_canonico = fun.canonicalizar_planes(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)

df_can["Nombre_de_archivo"] = nombre_canonico

Nombre original : Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
Nombre canónico : SURA Planes 202511.xlsx


🧮 Bloque 8 — Eliminación de filas finales nulas (trailing totals)

In [ ]:
df_can = fun.drop_trailing_mostly_null(
    df_can,
    null_check_exclude=("Nombre_de_archivo",),
    also_exclude_money_cols=("PrimaBruta", "PrimaNeta", "Comision25"),
    null_ratio_threshold=0.80,
    verbose=True,
)

🧹 Última fila TOTAL detectada (sumas OK y muchos nulos). Eliminando…

🧾 Fila eliminada:


,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,Tasa,Planes,PrimaNeta,Comision25,ComisionVariable,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
258,,,,,,,,,,1254423,,,1054136.9747899151,263534.2436974788,None,,,,SURA Planes 202511.xlsx


🔢 Bloque 9 — Casting de tipos + preparación df_sql

In [ ]:
# --- Casting numérico ---
fun.cast_numeric_columns(
    df_can,
    bigint_cols=["foliocredito", "FolioOrigen"],
    int_cols=["rutafiliado", "Plazo", "fechaPrimerVto", "FechaUltimoVto", "FechaOrigen"],
    float_cols=["MontoBruto", "ValorCuota", "PrimaBruta", "PrimaNeta", "Comision25", "Tasa", "ComisionVariable"],
)

# --- DV: char(1) mayúscula ---
fun.normalize_dv_column(df_can, "dvafiliado")

# --- Strings: trim y largo máximo ---
fun.trim_string_columns(df_can, {
    "NombreAfiliado": 510,
    "Producto": 510,
    "Planes": 510,
    "Nombre_de_archivo": 50,
})

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)

# --- Nulos en columnas críticas ---
fun.report_nulls(df_can, ["foliocredito", "rutafiliado", "dvafiliado", "FolioOrigen", "Nombre_de_archivo"])

# --- Construir df_sql ---
cols_sql = [
    "foliocredito", "NombreAfiliado", "rutafiliado", "dvafiliado", "Plazo", "MontoBruto",
    "fechaPrimerVto", "FechaUltimoVto", "ValorCuota", "PrimaBruta", "Tasa", "Planes", "PrimaNeta",
    "Comision25", "ComisionVariable", "Producto", "FolioOrigen", "FechaOrigen", "Nombre_de_archivo",
]

df_sql = fun.build_sql_frame(df_can, cols_sql)

print("\n📊 df_sql listo con columnas:", list(df_sql.columns))
fun.pretty_table(df_sql, n=5)

✅ dtypes DESPUÉS:
 foliocredito                  Int64
NombreAfiliado       string[python]
rutafiliado                   Int64
dvafiliado                   object
Plazo                         Int64
MontoBruto                  float64
fechaPrimerVto                Int64
FechaUltimoVto                Int64
ValorCuota                  float64
PrimaBruta                  float64
Tasa                        float64
Planes               string[python]
PrimaNeta                   float64
Comision25                  float64
ComisionVariable            float64
Producto             string[python]
FolioOrigen                   Int64
FechaOrigen                   Int64
Nombre_de_archivo    string[python]
dtype: object

🔎 Nulos en columnas críticas:
 - foliocredito: 0 nulos
 - rutafiliado: 0 nulos
 - dvafiliado: 0 nulos
 - FolioOrigen: 0 nulos
 - Nombre_de_archivo: 0 nulos

📦 df_sql listo con columnas: ['foliocredito', 'NombreAfiliado', 'rutafiliado', 'dvafiliado', 'Plazo', 'MontoBruto', 'fechaPri

C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_10812\2329058015.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s2 = s.astype(str).str.strip().replace({"": np.nan, "None": np.nan, "none": np.nan})


🧾 Bloque 10 — Extracción de valores de partición

In [ ]:
assert "Nombre_de_archivo" in df_sql.columns, "Falta la columna 'Nombre_de_archivo' en df_sql."

df_sql["Nombre_de_archivo"] = (
    df_sql["Nombre_de_archivo"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Nombre_de_archivo"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚿 No se encontraron valores válidos en 'Nombre_de_archivo'.")

counts = (
    df_sql["Nombre_de_archivo"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} Nombre_de_archivo distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 Nombre_de_archivo distintos en el df_sql:
   - SURA Planes 202511.xlsx: 258 filas


🔄 Bloque 11 — Carga a SQL Server (replace_partition / append)

In [ ]:
resumen = []

table_name = data["tablas_remotas"]["volvek_acumulado_planes"]

for nombre_archivo in vals:

    df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

    if df_sub.empty:
        print(f"⚠️ No se encontraron filas para Nombre_de_archivo = '{nombre_archivo}'. Se omite.")
        continue

    sql_count = f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table_name}
        WHERE Nombre_de_archivo = '{nombre_archivo}'
    """

    df_cnt = fun.query_to_df(
        sql=sql_count,
        connection_string=connection_string,
        engine="pandas",
        return_iter=False,
    )

    existing_count = int(df_cnt.iloc[0, 0]) if not df_cnt.empty else 0

    if existing_count > 0:
        print(
            f"♻️ Se encontró data previa para Nombre_de_archivo = "
            f"'{nombre_archivo}' ({existing_count} filas en {table_name})."
        )
        print("🧹 Eliminando filas anteriores para reemplazarlas...")

        summary = fun.df_to_db(
            df=df_sub,
            connection_string=connection_string,
            schema=schema,
            table=table_name,
            mode="replace_partition",
            partition_column="Nombre_de_archivo",
            partition_values=[nombre_archivo],
            engine="pandas",
        )

        accion = "reemplazo"
        filas_borradas = summary["rows_deleted"]
        print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {filas_borradas}")

    else:
        print(
            f"🆕 No hay data previa para Nombre_de_archivo = "
            f"'{nombre_archivo}'. Se insertará como archivo nuevo."
        )

        summary = fun.df_to_db(
            df=df_sub,
            connection_string=connection_string,
            schema=schema,
            table=table_name,
            mode="append",
            engine="pandas",
        )

        accion = "inserción nueva"

    filas_sub = len(df_sub)
    print(f"📥 Insertadas {filas_sub} filas para Nombre_de_archivo = '{nombre_archivo}'.")
    resumen.append((nombre_archivo, filas_sub, existing_count, accion))

print("\n📈 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")

🆕 No hay data previa para Nombre_de_archivo = 'SURA Planes 202511.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 258 filas para Nombre_de_archivo = 'SURA Planes 202511.xlsx'.

📊 Resumen de carga por Nombre_de_archivo:
   - SURA Planes 202511.xlsx: 258 filas cargadas (archivo nuevo).


🗑️ Bloque 12 — Eliminación de archivo origen

In [ ]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\VOLVEK PLANES\Base cesantia SURA Planes OCTUBRE 2025.xlsx


# SQL

## VALIDACIONES / INSPECCIÓN

### Bloque 13 — Query: archivos cargados

In [ ]:
query_1 = f"""
-- [VOLVEK_PLANES] Archivos cargados (desc)
SELECT DISTINCT Nombre_de_archivo
FROM {data["tablas_remotas"]["volvek_acumulado_planes"]}
ORDER BY Nombre_de_archivo DESC;
"""

df_q1 = fun.query_to_df(
    sql=query_1,
    connection_string=connection_string,
    engine="pandas",
)

fun.pretty_table(df_q1)

## CONSTRUCCIÓN DE BASES FINALES POR CORREDOR / FUENTE

### Bloque 14 — Query: DROP tabla final

In [ ]:
query_2 = f"""
DROP TABLE IF EXISTS {data["tablas_remotas"]["planes_final_acumulado"]};
"""

fun.exec_sql(query_2, connection_string)

### Bloque 15 — Query: SELECT INTO tabla final (CASE por tipo de Plan)

In [ ]:
query_3 = f"""
SELECT *,
       (CASE
            WHEN Planes IN ('Plan 03 Cuotas', 'Plan 3 Cuotas') THEN 4780715
            WHEN Planes IN ('Plan 04 Cuotas', 'Plan 4 Cuotas') THEN 4780716
            WHEN Planes IN ('Plan 06 Cuotas', 'Plan 6 Cuotas') THEN 4780717
            WHEN Planes IN ('Plan 08 Cuotas', 'Plan 8 Cuotas') THEN 4780718
            WHEN Planes = 'Plan 12 Cuotas' THEN 4780719
        END) AS POLIZA,
       SUBSTRING(Nombre_de_archivo,
                 LEN(Nombre_de_archivo) - CHARINDEX('.', REVERSE(Nombre_de_archivo)) - 5,
                 6) AS MES_Recaudacion,
       (CASE
            WHEN Planes IN ('Plan 03 Cuotas', 'Plan 3 Cuotas') THEN 4331
            WHEN Planes IN ('Plan 04 Cuotas', 'Plan 4 Cuotas') THEN 4422
            WHEN Planes IN ('Plan 06 Cuotas', 'Plan 6 Cuotas') THEN 4332
            WHEN Planes IN ('Plan 08 Cuotas', 'Plan 8 Cuotas') THEN 4333
            WHEN Planes = 'Plan 12 Cuotas' THEN 4334
        END) AS PLAN_TECNICO,
       (CASE
            WHEN Planes IN ('Plan 03 Cuotas', 'Plan 3 Cuotas') THEN 3
            WHEN Planes IN ('Plan 04 Cuotas', 'Plan 4 Cuotas') THEN 4
            WHEN Planes IN ('Plan 06 Cuotas', 'Plan 6 Cuotas') THEN 6
            WHEN Planes IN ('Plan 08 Cuotas', 'Plan 8 Cuotas') THEN 8
            WHEN Planes = 'Plan 12 Cuotas' THEN 12
        END) AS PLAZO_CUOTAS,
       'Credito Consumo' AS Negocio
INTO {data["tablas_remotas"]["planes_final_acumulado"]}
FROM {data["tablas_remotas"]["volvek_acumulado_planes"]};
"""

fun.exec_sql(query_3, connection_string)